# DG-TCAV 3D Latent Diffusion (Kaggle / Colab Launcher)

Trains the conditional 3D latent diffusion model (CN / MCI / AD, CFG null index 3) on **Train** subjects of the
Cohort A + B longitudinal superset and generates quality-gated TCAV probes. Dependency chain:

`audit_inputs -> train_ae -> validate_ae -> cache_latents -> train_ldm -> generate_probes -> export_probe_features`

* Test subjects (the 247 master test IDs) are a denylist: their scans are never opened, cached or sampled.
* ADNI data and participant-linked probes must stay in authorized storage. Check the ADNI agreement before using a hosted runtime.
* Pick the GPU profile (`t4`, `p100`, `a100`) in section 1. Always run each `--preflight` first.

## 1. Platform, GPU profile and paths

In [ ]:
import os, sys, subprocess
PROFILE = "t4"          # t4 | p100 | a100  (blueprint section 4)
BRANCH = "feat/3d-latent-diffusion"
REPO = "https://github.com/Alishams9008/dg-tcav-classifier-fyp.git"
ON_KAGGLE = os.path.isdir("/kaggle/working")
if ON_KAGGLE:
    WORK = "/kaggle/working"
    DRIVE = "/kaggle/input/dg-tcav-fyp"          # attach your private dataset here
else:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content"
    DRIVE = "/content/drive/MyDrive/DG-TCAV_FYP_NUST"
OUTPUTS = f"{DRIVE}/DG_TCAV_OUTPUTS" if not ON_KAGGLE else f"{WORK}/outputs_in"
RUN_DIR = f"{DRIVE}/DG_TCAV_OUTPUTS/diffusion_outputs" if not ON_KAGGLE else f"{WORK}/diffusion_outputs"
print("profile:", PROFILE, "| kaggle:", ON_KAGGLE)

## 2. Clone repository and install dependencies

In [ ]:
%cd {WORK}
!git clone --branch {BRANCH} {REPO} dg-tcav-classifier-fyp
%cd dg-tcav-classifier-fyp
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), "GiB")
!git rev-parse HEAD

## 3. Copy scans and write the run configuration

Copies the preprocessed `*_preprocessed_128.nii.gz` scans and companion masks to local disk and writes a small
override config that points the repository config at your Drive files. Edit the paths if your layout differs.

In [ ]:
import yaml, pathlib
LOCAL = f"{WORK}/preprocessed_128"
!mkdir -p {LOCAL}
!rsync -a --info=progress2 --include='*_preprocessed_128*.nii.gz' --exclude='*' {OUTPUTS}/preprocessed_128/ {LOCAL}/
!find {LOCAL} -maxdepth 2 -name '*_preprocessed_128.nii.gz' | wc -l

SETUP = f"{OUTPUTS}/classifier_setup"
override = {
    "extends": "diffusion_ldm_128.yaml",
    "experiment": {"output_dir": RUN_DIR, "profile": PROFILE},
    "data": {
        "master_split_path": f"{SETUP}/cohort_a_splits.json",
        "longitudinal_manifest_path": f"{DRIVE}/03_Cohort_B_Diffusion_Longitudinal/cohort_b_longitudinal_manifest.csv",
        "qc_index_path": f"{RUN_DIR}/diffusion_qc_index.csv",
        "volume_root": LOCAL,
        "index_path": f"{RUN_DIR}/diffusion_index.json",
    },
    "latent": {"cache_dir": f"{RUN_DIR}/latent_cache"},
    "atlas": {"cerebra_path": f"{DRIVE}/atlas/CerebrA_MNI152_2009c_sym.nii.gz",
              "output_dir": f"{RUN_DIR}/atlas_masks_128"},
    "probes": {"output_dir": f"{RUN_DIR}/probes", "export_dir": f"{RUN_DIR}/probe_export"},
}
pathlib.Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
CFG = "configs/diffusion_colab.yaml"
pathlib.Path(CFG).write_text(yaml.safe_dump(override, sort_keys=False))
print(open(CFG).read())

## 4. audit_inputs

Builds the visit-keyed QC CSV for the preprocessed scans that exist today (1,593 baseline scans; more as the Cohort B
follow-ups are preprocessed), then the leak-free Train/Val index. The audit prints counts, exclusion reasons and the
master-test denylist size. It never resolves a test scan.

In [ ]:
!python -m src.diffusion.index_diffusion qc-index \
  --baseline-manifest "{SETUP}/cohort_a_baseline_manifest.csv" \
  --processing-manifest "{SETUP}/preprocessing_manifest.csv" \
  --volume-root {LOCAL} --output {RUN_DIR}/diffusion_qc_index.csv --layout baseline
!python -m src.diffusion.index_diffusion audit --config {CFG}

### 4b. Atlas masks (one time)

Needs the labeled symmetric 2009c CerebrA volume. Pin its SHA-256 in `atlas.cerebra_sha256` or pass it below.

In [ ]:
import hashlib, glob
ATLAS = f"{DRIVE}/atlas/CerebrA_MNI152_2009c_sym.nii.gz"
if os.path.exists(ATLAS):
    digest = hashlib.sha256(open(ATLAS, "rb").read()).hexdigest()
    print("atlas sha256:", digest)
    scan = sorted(glob.glob(f"{LOCAL}/*_preprocessed_128.nii.gz"))[0]
    !python -m src.diffusion.build_atlas_masks {ATLAS} template/mni152.nii.gz {scan} {RUN_DIR}/atlas_masks_128 --atlas-sha256 {digest}
else:
    print("Upload the CerebrA label volume to", ATLAS)

## 5. train_ae (Stage 1)

Preflight first: 1 + 20 updates plus a validation decode, then check `preflight.json` (`admitted` must be true).

In [ ]:
!python -m src.diffusion.train_stage1_ae --config {CFG} --profile {PROFILE} --preflight
!cat {RUN_DIR}/stage1_ae/preflight.json

In [ ]:
!python -m src.diffusion.train_stage1_ae --config {CFG} --profile {PROFILE}   # resumes from last.pt automatically

## 6. validate_ae

Validation PSNR / SSIM of `decode(mu)` on the validation subjects against the project targets (28.5 dB, 0.88).

In [ ]:
!python -m src.diffusion.train_stage1_ae --config {CFG} --profile {PROFILE} --validate-only

## 7. cache_latents

Freezes the AE, fits float64 Train-only latent offset/scale and caches scaled `[4,16,16,16]` latents.

In [ ]:
!python -m src.diffusion.cache_latents --config {CFG} --profile {PROFILE}

## 8. train_ldm (Stage 2)

In [ ]:
!python -m src.diffusion.train_stage2_ldm --config {CFG} --profile {PROFILE} --preflight
!cat {RUN_DIR}/stage2_ldm/preflight.json

In [ ]:
!python -m src.diffusion.train_stage2_ldm --config {CFG} --profile {PROFILE}   # EMA 0.999, CFG dropout 10 % / 15 %

## 9. generate_probes

Targets 200 accepted unique volumes for each of HA, ECT, VE, THD, GCT, AA and PBM (1,400 total). Without a validated morphometry
table the run is class-only and every acceptance is labeled `exploratory`. Shortfalls are reported in `generation_summary.json`.

In [ ]:
!python -m src.diffusion.generate_tcav_probes run --config {CFG} --profile {PROFILE} --atlas-dir {RUN_DIR}/atlas_masks_128
!cat {RUN_DIR}/probes/generation_summary.json

## 10. export_probe_features

Builds a classifier-style view of the accepted probes and runs the existing activation exporter against a frozen checkpoint.
Set `CLASSIFIER_CONFIG` and `CHECKPOINT` to the exact classifier whose layer-4 features the TCAV step will use.

In [ ]:
CLASSIFIER_CONFIG = "configs/colab_medicalnet_resnet18_128.yaml"
CHECKPOINT = f"{OUTPUTS}/classifier_outputs/REPLACE_WITH_YOUR_ACTUAL_RUN_NAME/best.pt"
!python -m src.diffusion.generate_tcav_probes export --config {CFG} --classifier-config {CLASSIFIER_CONFIG}
!python -m src.export_activations --config {RUN_DIR}/probe_export/probe_export_config.yaml \
  --checkpoint "{CHECKPOINT}" --split train --output {RUN_DIR}/probe_features